In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [2]:
customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")
orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")
items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")
payments = pd.read_csv("../data/raw/olist_order_payments_dataset.csv")
reviews = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv")
products = pd.read_csv("../data/raw/olist_products_dataset.csv")
sellers = pd.read_csv("../data/raw/olist_sellers_dataset.csv")
translation = pd.read_csv("../data/raw/product_category_name_translation.csv")

In [3]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col])

reviews["review_creation_date"] = pd.to_datetime(reviews["review_creation_date"])
reviews["review_answer_timestamp"] = pd.to_datetime(reviews["review_answer_timestamp"])

items["shipping_limit_date"] = pd.to_datetime(items["shipping_limit_date"])

In [4]:
products = products.merge(
    translation,
    on="product_category_name",
    how="left"
)

products.rename(
    columns={"product_category_name_english": "category"},
    inplace=True
)

products.drop(columns=["product_category_name"], inplace=True)

products["category"] = products["category"].fillna("Unknown")

products = products.dropna(
    subset=[
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ]
)

Orders Feature Engineering

In [5]:
orders["purchase_year"] = orders["order_purchase_timestamp"].dt.year

In [6]:
orders["purchase_month"] = orders["order_purchase_timestamp"].dt.month

In [7]:
orders["purchase_month_name"] = orders["order_purchase_timestamp"].dt.month_name()

In [8]:
orders["purchase_quarter"] = orders["order_purchase_timestamp"].dt.quarter

In [9]:
orders["purchase_day"] = orders["order_purchase_timestamp"].dt.day

In [10]:
orders["purchase_weekday"] = orders["order_purchase_timestamp"].dt.day_name()

In [11]:
orders["purchase_hour"] = orders["order_purchase_timestamp"].dt.hour

In [12]:
orders["is_weekend"] = (
    orders["purchase_weekday"]
    .isin(["Saturday", "Sunday"])
    .astype(int)
)

In [13]:
orders["delivery_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_purchase_timestamp"]
).dt.days

In [14]:
orders["estimated_delivery_days"] = (
    orders["order_estimated_delivery_date"]
    - orders["order_purchase_timestamp"]
).dt.days

In [15]:
orders["delivery_delay"] = (
    orders["order_delivered_customer_date"]
    - orders["order_estimated_delivery_date"]
).dt.days

In [16]:
orders["is_late"] = (
    orders["delivery_delay"] > 0
).astype(int)

Review Features

In [17]:
conditions = [
    reviews["review_score"] >= 4,
    reviews["review_score"] == 3,
    reviews["review_score"] <= 2
]

choices = [
    "Positive",
    "Neutral",
    "Negative"
]

reviews["review_sentiment"] = np.select(
    conditions,
    choices,
    default="Unknown"
)

In [18]:
reviews["positive_review"] = (
    reviews["review_score"] >= 4
).astype(int)

In [19]:
reviews["neutral_review"] = (
    reviews["review_score"] == 3
).astype(int)

In [20]:
reviews["negative_review"] = (
    reviews["review_score"] <= 2
).astype(int)

Product Features

In [21]:
products["product_volume_cm3"] = (
    products["product_length_cm"]
    * products["product_width_cm"]
    * products["product_height_cm"]
)

In [22]:
products["product_weight_kg"] = (
    products["product_weight_g"] / 1000
)

Payment Features

In [23]:
payments["installment_type"] = np.where(
    payments["payment_installments"] == 1,
    "Single Payment",
    "Installments"
)

In [24]:
payments["high_installment"] = (
    payments["payment_installments"] >= 6
).astype(int)